# ForgeLM — Full 3-Stage Training on Kaggle (2×T4)

**Stages:** DAPT → SFT → DPO on `Qwen/Qwen2.5-1.5B-Instruct`

**Eval:** all 4 checkpoints (base / dapt / sft / dpo) on 300 held-out examples

**Expected runtime:** ~8–10h on T4×2


In [ ]:
# ── Cell 1: Install deps ──────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q',
    'torch>=2.1','transformers>=4.40','peft>=0.11','trl>=0.9',
    'bitsandbytes>=0.43','datasets>=2.19','accelerate>=0.30',
    'pydantic>=2.5','structlog>=23.0','scipy>=1.11',
    'anthropic>=0.40.0',  # only needed if generating data in-notebook
], check=True)
print('✓ deps installed')

In [ ]:
# ── Cell 2: Setup paths + clone repo ─────────────────────────────────────
import os, sys, subprocess
from pathlib import Path

WORKING = Path('/kaggle/working')
REPO = WORKING / 'forgelm'
REPO_URL = 'https://github.com/deepanshu-s18/forgelm.git'

if not REPO.exists():
    result = subprocess.run(
        ['git', 'clone', '--depth=1', REPO_URL, str(REPO)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('Clone failed:', result.stderr)
        print('Trying with GITHUB_TOKEN secret...')
        token = os.environ.get('GITHUB_TOKEN', '')
        url_with_token = f'https://{token}@github.com/deepanshu-s18/forgelm.git'
        subprocess.run(['git', 'clone', '--depth=1', url_with_token, str(REPO)], check=True)

sys.path.insert(0, str(REPO))
os.chdir(REPO)
print('✓ repo ready at', REPO)


In [ ]:
# ── Cell 3: Unpack Kaggle dataset → data/ dirs ───────────────────────────
import shutil, zipfile
from pathlib import Path

# Show what's in the input dataset
DATA_SRC = Path('/kaggle/input/forgelm-data')
print('Dataset contents:')
all_files = list(DATA_SRC.rglob('*'))
for f in sorted(all_files)[:30]:
    print(' ', f.relative_to(DATA_SRC))

# ── Step 1: Unzip any zip files found in the dataset root ─────────────────
EXTRACT = Path('/kaggle/working/extracted')
EXTRACT.mkdir(exist_ok=True)

for zf in DATA_SRC.rglob('*.zip'):
    print(f'Extracting {zf.name}...')
    with zipfile.ZipFile(zf) as z:
        z.extractall(EXTRACT)

# ── Step 2: Map files into data/ regardless of source structure ────────────
# Look in both the dataset root and extracted dir
SEARCH_ROOTS = [DATA_SRC, EXTRACT]

def find_file(name):
    for root in SEARCH_ROOTS:
        for p in root.rglob(name):
            return p
    return None

# Map: destination path → source filename
COPY_MAP = {
    'data/sft/sft_toolcall.jsonl': 'sft_toolcall.jsonl',
    'data/events/sft_events.jsonl': 'sft_events.jsonl',
    'data/dpo/dpo_pairs.jsonl': 'dpo_pairs.jsonl',
    'eval/eval_data.jsonl': 'eval_data.jsonl',
}

for dst_rel, src_name in COPY_MAP.items():
    dst = Path(dst_rel)
    dst.parent.mkdir(parents=True, exist_ok=True)
    src = find_file(src_name)
    if src:
        shutil.copy(src, dst)
        lines = sum(1 for _ in open(dst))
        print(f'  ✓ {dst_rel}: {lines} lines')
    else:
        print(f'  ✗ {src_name} not found!')

# ── Step 3: Copy corpus txt files ─────────────────────────────────────────
corpus_dst = Path('data/corpus')
corpus_dst.mkdir(parents=True, exist_ok=True)
corpus_count = 0
for root in SEARCH_ROOTS:
    for txt in root.rglob('*.txt'):
        shutil.copy(txt, corpus_dst / txt.name)
        corpus_count += 1
# Also copy manifest
manifest = find_file('manifest.json')
if manifest:
    shutil.copy(manifest, corpus_dst / 'manifest.json')
print(f'  ✓ data/corpus: {corpus_count} txt files')
print('\n✓ Data setup complete.')


In [ ]:
# ── Cell 4: Verify GPU + data ─────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from pathlib import Path
corpus_files = list(Path('data/corpus').glob('*.txt'))
sft_lines = sum(1 for l in open('data/sft/sft_toolcall.jsonl') if l.strip()) if Path('data/sft/sft_toolcall.jsonl').exists() else 0
dpo_lines  = sum(1 for l in open('data/dpo/dpo_pairs.jsonl')   if l.strip()) if Path('data/dpo/dpo_pairs.jsonl').exists()   else 0
print(f'Corpus chunks: {len(corpus_files)}')
print(f'SFT samples:   {sft_lines}')
print(f'DPO pairs:     {dpo_lines}')

## Stage 1 — Domain-Adaptive Pre-Training (DAPT)

Trains on EDGAR corpus. ~3h on T4.

In [ ]:
# ── Cell 5: Stage 1 DAPT ─────────────────────────────────────────────────
import subprocess
result = subprocess.run([
    'python', '-m', 'forgelm.training.stage1_dapt',
    '--corpus', 'data/corpus',
    '--output-dir', '/kaggle/working/checkpoints',
    '--seed', '42',
    '--epochs', '1',
    '--batch-size', '4',
], capture_output=False)
print('Stage 1 return code:', result.returncode)

In [ ]:
# ── Cell 6: Verify Stage 1 loss ──────────────────────────────────────────
import json
from pathlib import Path
metrics = json.loads(Path('/kaggle/working/checkpoints/stage1_dapt/metrics.json').read_text())
print('Stage 1 metrics:', json.dumps(metrics, indent=2))
assert metrics['train_loss'] < 3.0, f'Loss too high: {metrics["train_loss"]} — check corpus loading'

## Stage 2 — Supervised Fine-Tuning (SFT)

Tool-call sequences (70%) + event extraction (30%). ~2h.

In [ ]:
# ── Cell 7: Stage 2 SFT ──────────────────────────────────────────────────
result = subprocess.run([
    'python', '-m', 'forgelm.training.stage2_sft',
    '--stage1-dir', '/kaggle/working/checkpoints/stage1_dapt/final',
    '--sft-toolcall', 'data/sft/sft_toolcall.jsonl',
    '--sft-events', 'data/events/sft_events.jsonl',
    '--output-dir', '/kaggle/working/checkpoints',
    '--seed', '42', '--epochs', '2',
], capture_output=False)
print('Stage 2 return code:', result.returncode)

In [ ]:
# ── Cell 8: Verify Stage 2 loss ──────────────────────────────────────────
metrics2 = json.loads(Path('/kaggle/working/checkpoints/stage2_sft/metrics.json').read_text())
print('Stage 2 metrics:', json.dumps(metrics2, indent=2))
assert metrics2['train_loss'] < metrics['train_loss'], 'SFT should lower loss vs DAPT'

## Stage 3 — DPO (Direct Preference Optimisation)

β=0.1, 4 error families (wrong_ticker / missing_correction / hallucinated_date / vague). ~1.5h.

In [ ]:
# ── Cell 9: Stage 3 DPO ──────────────────────────────────────────────────
result = subprocess.run([
    'python', '-m', 'forgelm.training.stage3_dpo',
    '--stage2-dir', '/kaggle/working/checkpoints/stage2_sft/final',
    '--dpo-pairs', 'data/dpo/dpo_pairs.jsonl',
    '--output-dir', '/kaggle/working/checkpoints',
    '--seed', '42', '--epochs', '1',
], capture_output=False)
print('Stage 3 return code:', result.returncode)

## Evaluation — All 4 Checkpoints

Eval on 300 held-out decontaminated examples. Produces M1–M4 + McNemar + CI.

In [ ]:
# ── Cell 10: Eval all checkpoints ────────────────────────────────────────
import json
from pathlib import Path

BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
EVAL_DATA  = 'eval/eval_data.jsonl'
RESULTS = {}

checkpoints = {
    'base': BASE_MODEL,
    'dapt': '/kaggle/working/checkpoints/stage1_dapt/final',
    'sft':  '/kaggle/working/checkpoints/stage2_sft/final',
    'dpo':  '/kaggle/working/checkpoints/stage3_dpo/final',
}

for name, ckpt in checkpoints.items():
    print(f'\n=== Evaluating {name} ===')
    out = Path(f'/kaggle/working/eval_{name}.json')
    result = subprocess.run([
        'python', '-m', 'forgelm.eval.run_eval',
        '--model', ckpt,
        '--eval-data', EVAL_DATA,
        '--n-eval', '200',
        '--seed', '42',
        '--out', str(out),
    ], capture_output=False)
    if out.exists():
        RESULTS[name] = json.loads(out.read_text())
        m = RESULTS[name].get('metrics', {})
        print(f'  M1: {m.get("schema_validity_rate",{}).get("mean","?")}  '
              f'M2: {m.get("tool_call_accuracy",{}).get("mean","?")}  '
              f'M3: {m.get("correction_recall",{}).get("mean","?")}  '
              f'M4: {m.get("hallucination_rate",{}).get("mean","?")}')  

# 3-seed stability on final model
print('\n=== 3-seed stability (dpo) ===')
result = subprocess.run([
    'python', '-m', 'forgelm.eval.run_eval',
    '--model', checkpoints['dpo'],
    '--eval-data', EVAL_DATA,
    '--n-eval', '200', '--seeds',
    '--out', '/kaggle/working/eval_dpo_seeds.json',
], capture_output=False)
if Path('/kaggle/working/eval_dpo_seeds.json').exists():
    RESULTS['dpo_seeds'] = json.loads(Path('/kaggle/working/eval_dpo_seeds.json').read_text())

# Save combined
combined_path = Path('/kaggle/working/eval_all.json')
combined_path.write_text(json.dumps(RESULTS, indent=2))
print('\n✓ All evals done →', combined_path)

In [ ]:
# ── Cell 11: Generate stats report + results.md ───────────────────────────
Path('eval').mkdir(exist_ok=True)
Path('/kaggle/working/eval_all.json').replace(Path('eval/results_raw.json'))
result = subprocess.run([
    'python', '-m', 'forgelm.eval.stats_report',
    '--results', 'eval/results_raw.json',
    '--out', '/kaggle/working/results.md',
], capture_output=False)
print(Path('/kaggle/working/results.md').read_text())

In [ ]:
# ── Cell 12: Package outputs for download ─────────────────────────────────
import zipfile, os
from pathlib import Path

output_zip = Path('/kaggle/working/forgelm_results.zip')
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in Path('/kaggle/working').glob('*.json'):
        zf.write(f, f.name)
    for f in Path('/kaggle/working').glob('*.md'):
        zf.write(f, f.name)
    # Include final checkpoint adapter weights only (not full model)
    for f in Path('/kaggle/working/checkpoints/stage3_dpo/final').glob('adapter_*'):
        zf.write(f, f'checkpoints/stage3_dpo/final/{f.name}')

print(f'✓ Results packaged: {output_zip} ({output_zip.stat().st_size/1e6:.1f} MB)')
print('Download forgelm_results.zip from the Kaggle output panel →')
print('  eval/results.md: copy into forgelm/eval/results.md')
print('  eval_all.json:   copy into forgelm/eval/results.json')
print('  checkpoints/:    push adapter weights to HuggingFace Hub')

## After the Notebook Finishes

1. **Download** `forgelm_results.zip` from the Kaggle output panel
2. **Copy** `eval/results.md` → `forgelm/eval/results.md` in your local repo
3. **Update** `forgelm/README.md` eval table with real numbers
4. **Push** adapter weights to HuggingFace Hub: `deepanshusingh/forgelm-dpo`
5. **Run** tests again: `pytest tests/ -q` (should still be 23/23)
6. **Commit**: `git commit -m 'feat: add real training results from Kaggle T4'`
